# Physical variables → components → scores → bins

This notebook follows the highest-level public workflow without hiding it inside an example runner. A two-variable event model has three additive intensity components. FisherBin evaluates the components, constructs coefficient scores, and learns eight hard bins.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import scorequant as fb
from examples.linear_workflow import background_1, background_2, signal

rng = np.random.default_rng(17)
X_mc = np.column_stack([rng.uniform(0, 1, 2_000), rng.uniform(-1, 1, 2_000)])
mc_weights = 0.5 + rng.random(len(X_mc))
X_mc.shape

## Declare the scientific model

For $\lambda(x;c)=\sum_k c_k\phi_k(x)$, the coefficient score is $s_k(x)=\phi_k(x)/\lambda(x;c_0)$. `LinearComponents` stores named component callables, their reference coefficients, and variable names so the fitted object can accept physical variables later.

In [ ]:
model = fb.LinearComponents(
    components={"signal": signal, "background_1": background_1, "background_2": background_2},
    coefficients={"signal": 1.0, "background_1": 0.4, "background_2": 0.2},
    variables=["energy", "cos_theta"],
)
component_values = np.asarray(model.evaluate_components(X_mc))
component_values.shape, component_values[:2]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
for index, (axis, name) in enumerate(zip(axes, model.component_names, strict=True)):
    points = axis.scatter(X_mc[:, 0], X_mc[:, 1], c=component_values[:, index], s=6, cmap="viridis")
    axis.set(title=name, xlabel="energy", ylabel="cos(theta)")
    fig.colorbar(points, ax=axis)

## Fit from physical variables

`fit` owns the complete physical-variable workflow. Use `fit_components` instead when component values have already been evaluated, or `fit_scores` when the score construction belongs entirely to the application.

In [ ]:
result = fb.fit(
    X_mc,
    model=model,
    weights=mc_weights,
    n_bins=8,
    config=fb.KMeansConfig(seed=42, n_init=4),
)
print(result.report())

In [ ]:
X_data = np.column_stack([rng.uniform(0, 1, 500), rng.uniform(-1, 1, 500)])
data_labels = np.asarray(result.predict(X_data))
counts = np.bincount(data_labels, minlength=result.n_bins)
counts

## Output contract

Because the fit started from physical variables, `result.predict` also accepts physical variables and applies the frozen model internally. The eight counts are ready for an application likelihood; fitting the three coefficients from those counts is deliberately outside FisherBin.